In [16]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import bootstrap
from tqdm import tqdm
import wandb
import tempfile
import os
import torch as th
from pprint import pprint

tqdm.pandas()

In [17]:
metric_cols = [
    "cos_sim",
    "cos_sim_sqrt",
    "js_sim"
]

In [18]:
data = th.load("/scratch/hdd001/home/adamo/frag-gnn/inference_outputs/msg/benchmark/fragnnet_msg/test.pkl")

In [19]:
pprint(data.keys())

dict_keys(['true_mzs', 'true_logprobs', 'true_batch_idxs', 'pred_mzs', 'pred_logprobs', 'pred_batch_idxs', 'spec_ce', 'ios_ce', 'oos_ce', 'primary_loss', 'null_formula_prob', 'oos_prob', 'loss', 'cos_sim_0.01', 'jss_0.1', 'cos_hun', 'cos_sim_sqrt_0.01', 'jss_hun', 'cos_sim_0.1', 'jss_0.01', 'cos_sim_sqrt_0.1', 'jss_sqrt_0.01', 'jss_sqrt_0.1'])


In [20]:
num_datapoints = len(data["cos_sim_0.01"])
df = {}
df["cos_sim"] = data["cos_sim_0.01"].numpy()
df["js_sim"] = data["jss_0.01"].numpy()
df["cos_sim_sqrt"] = data["cos_sim_sqrt_0.01"].numpy()
df["method"] = np.array(["fragnnet" for _ in range(num_datapoints)])
df = pd.DataFrame(df)
df

,cos_sim,js_sim,cos_sim_sqrt,method
0,0.948707,0.881730,0.850913,fragnnet
1,0.992814,0.924094,0.893146,fragnnet
2,0.848841,0.838364,0.802144,fragnnet
3,0.311855,0.628174,0.549550,fragnnet
4,0.744671,0.694117,0.589568,fragnnet
...,...,...,...,...
9785,0.086817,0.451682,0.292486,fragnnet
9786,0.264807,0.548449,0.413264,fragnnet
9787,0.509817,0.691667,0.604124,fragnnet
9788,0.394514,0.613534,0.496824,fragnnet


In [23]:
def calculate_ci(df, metric_cols):

    # Calculate means for all metrics into a single table
    df_mean = df.groupby('method')[metric_cols].mean().round(2)

    # Calculate confidence intervals for all metrics into a single table
    def get_ci(col_vals, confidence_level=0.999, n_resamples=20_000):
        res = bootstrap((col_vals,), np.mean, confidence_level=confidence_level, n_resamples=n_resamples)
        ci = res.confidence_interval
        return f'{ci.low:.2f}-{ci.high:.2f}'
    def get_ci_for_each_col(df_method):
        return df_method.apply(get_ci, axis=0)
    df_ci = df.groupby('method')[metric_cols].progress_apply(lambda df_method: get_ci_for_each_col(df_method))

    # Merge tables with means and confidence intervals
    for col in metric_cols:
        df_mean[col] = df_mean[col].astype(str) + ' (' + df_ci[col] + ')'
    return df_mean

In [24]:
df_ci = calculate_ci(df, metric_cols)
df_ci

100%|██████████| 1/1 [00:14<00:00, 14.77s/it]


,cos_sim,cos_sim_sqrt,js_sim
method,,,
fragnnet,0.53 (0.52-0.54),0.52 (0.51-0.52),0.64 (0.63-0.64)
